# Project 1 — Sentiment Analysis: Fine-tune BERT

**Duration:** 5 min

## Setup & Tokenize

In [ ]:
from datasets import load_dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch

dataset = load_dataset('amazon_polarity')
# Use 10k train, 2k test for speed
train = dataset['train'].shuffle(seed=42).select(range(10000))
test  = dataset['test'].shuffle(seed=42).select(range(2000))

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['content'], truncation=True, padding='max_length', max_length=128)

train = train.map(tokenize, batched=True)
test  = test.map(tokenize, batched=True)
train.set_format('torch', columns=['input_ids','attention_mask','label'])
test.set_format('torch', columns=['input_ids','attention_mask','label'])

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/ai-projects-intermediate/ip-project1-build.ipynb)


## Fine-tune

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': accuracy_score(p.label_ids, preds)}

args = TrainingArguments(
    output_dir='./sentiment-model',
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=100,
)

trainer = Trainer(model=model, args=args, train_dataset=train,
                  eval_dataset=test, compute_metrics=compute_metrics)
trainer.train()

## Inference Function

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    label = 'positive' if probs[1] > probs[0] else 'negative'
    return {'label': label, 'confidence': round(probs.max().item(), 3)}

print(predict_sentiment('This product is absolutely amazing, works perfectly!'))
print(predict_sentiment('Terrible quality, broke after one day.'))

```
{'label': 'positive', 'confidence': 0.998}
{'label': 'negative', 'confidence': 0.995}
```

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ Why do we use DistilBERT instead of full BERT for this project?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4372734912" value="0">
      <span>DistilBERT is more accurate</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4372734912" value="1">
      <span>DistilBERT is 40% smaller and 60% faster with ~97% of BERT's performance</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4372734912" value="2">
      <span>BERT doesn't support classification</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4372734912" value="3">
      <span>DistilBERT supports more languages</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>